# Trazabilidad de un agente con herramientas

En este notebook construimos un agente con herramientas y registramos cada llamada para analizar su comportamiento interno.

## Objetivos
- Comprender qué es una traza y un span en sistemas de observabilidad.
- Ver cómo se jerarquiza la ejecución de un agente con herramientas.
- Detectar pasos innecesarios, herramientas mal elegidas y errores silenciosos.

## Actividad práctica
- Crear un agente con 3 herramientas: calculadora, buscador local y consulta a CSV.
- Registrar cada tool call y cada decisión del agente.
- Visualizar la ejecución como tabla.

In [4]:
!pip install pandas langchain langchain-openai wikipedia

  Using cached pandas-3.0.6-cp315-cp315-win_amd64.whl.metadata (19 kB)
  Using cached langchain-1.4.5-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.7.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached numpy-2.5.4-cp315-cp315-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached langchain_core-1.6.10-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.15-py3-none-any.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.35-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.8-py3-none-any.whl.metadata (22 kB)
  Using cached pyyaml-6.0.3.tar.gz (130 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished w

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [29 lines of output]
      Traceback (most recent call last):
        File "C:\Users\river\AppData\Local\Programs\Python\Python315\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "C:\Users\river\AppData\Local\Programs\Python\Python315\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\river\AppData\Local\Programs\Python\Python315\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\river\AppData\Local\Temp\pip-build-env-9oxk6z8j\overlay\Lib\site-packa

In [5]:
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
import csv
import os
import time
import uuid
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

from langchain_core.messages import HumanMessage

# Creamos un archivo CSV local de ejemplo para la herramienta de consulta
csv_path = 'datos_locales.csv'
with open(csv_path, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['id', 'tema', 'respuesta'])
    writer.writeheader()
    writer.writerow({'id': 1, 'tema': 'clima', 'respuesta': 'Hoy estará soleado con temperaturas suaves.'})
    writer.writerow({'id': 2, 'tema': 'matemáticas', 'respuesta': 'La raíz cuadrada de 16 es 4.'})
    writer.writerow({'id': 3, 'tema': 'historia', 'respuesta': 'La independencia se celebró en 1810.'})

# Herramientas disponibles para el agente
def calculadora_tool(expression):
    # Herramienta simple que evalúa expresiones aritméticas seguras
    try:
        value = eval(expression, {'__builtins__': None}, {})
        return str(value), None
    except Exception as exc:
        return None, str(exc)


def buscador_local(query, knowledge_base):
    # Busca coincidencias en una lista local de documentos
    matches = [item for item in knowledge_base if query.lower() in item.lower()]
    return matches[:3] or ['No se encontró información relevante.']


def consulta_csv(topic, csv_file):
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if topic.lower() in row['tema'].lower():
                return row['respuesta']
    return 'No hay datos relevantes en el CSV.'


def call_llm(prompt_text):
    if llm is None:
        raise RuntimeError("El LLM no está configurado. Revisa GITHUB_BASE_URL y GITHUB_TOKEN.")

    def extract_text(response):
        if hasattr(response, "content"):
            return response.content
        if hasattr(response, "text"):
            return response.text
        if hasattr(response, "generations") and response.generations:
            return response.generations[0][0].text
        return str(response)

    try:
        message = HumanMessage(content=prompt_text)
        if hasattr(llm, "predict_messages"):
            response = llm.predict_messages([message])
        elif hasattr(llm, "generate"):
            response = llm.generate([[message]])
        else:
            response = llm([message])

        text = extract_text(response)
        return {
            'text': text,
            'model': getattr(llm, 'model_name', getattr(llm, 'model', 'gpt-4o')),
            'prompt_tokens': 0,
            'completion_tokens': 0,
            'latency': 0.0,
            'cost': 0.0,
            'error': None,
        }
    except Exception as e:
        return {
            'text': '',
            'model': getattr(llm, 'model_name', getattr(llm, 'model', 'gpt-4o')),
            'prompt_tokens': 0,
            'completion_tokens': 0,
            'latency': 0.0,
            'cost': 0.0,
            'error': str(e),
        }


In [ ]:
# Crear agente y ejecutar ejemplos de trazabilidad
if llm is None:
    raise RuntimeError("El LLM no está configurado. Verifica las credenciales y las variables de entorno.")

class ToolAgentTrace:
    def __init__(self):
        self.steps = []

    def add_step(self, step, tipo, input_data, output_data, latency, tokens, estado):
        self.steps.append({
            'step': step,
            'tipo': tipo,
            'input': input_data,
            'output': output_data,
            'latencia': f'{latency:.2f}s',
            'tokens': tokens,
            'estado': estado,
        })

    def dataframe(self):
        return pd.DataFrame(self.steps)

    def display(self):
        display(self.dataframe())

class ToolAgent:
    def __init__(self, llm, csv_path):
        self.llm = llm
        self.csv_path = csv_path
        self.knowledge_base = [
            'El sol sale por el este y se pone por el oeste.',
            'Python es un lenguaje de programación versátil.',
            'Los agentes de observabilidad deben registrar decisiones.',
        ]

    def _generate(self, prompt):
        return call_llm(prompt)

    def run(self, user_question):
        trace = ToolAgentTrace()
        current_step = 1

        prompt_decision = f"Decide qué herramienta usar para: {user_question}"
        start = time.time()
        decision = self._generate(prompt_decision)
        end = time.time()
        trace.add_step(current_step, 'llm', prompt_decision, decision['text'], end - start, 30, 'ok' if decision['error'] is None else 'error')
        current_step += 1

        tool_output = None
        status = 'ok'

        decision_text = decision['text'].lower()

        if 'calculadora' in decision_text or 'calcula' in decision_text:
            expression = ''.join([c for c in user_question if c.isdigit() or c in '+-*/. '])
            start = time.time()
            tool_output, error = calculadora_tool(expression)
            end = time.time()
            status = 'ok' if error is None else 'error'
            trace.add_step(current_step, 'tool', expression, tool_output or error, end - start, 0, status)
            current_step += 1

        elif 'buscar' in decision_text or 'información' in decision_text:
            start = time.time()
            results = buscador_local(user_question, self.knowledge_base)
            end = time.time()
            trace.add_step(current_step, 'tool', user_question, results, end - start, 0, 'ok')
            current_step += 1
            tool_output = '\n'.join(results)

        elif 'csv' in decision_text or 'archivo' in decision_text:
            topic = user_question.split()[-1] if user_question.split() else user_question
            start = time.time()
            tool_output = consulta_csv(topic, self.csv_path)
            end = time.time()
            trace.add_step(current_step, 'tool', topic, tool_output, end - start, 0, 'ok')
            current_step += 1

        if tool_output is not None:
            final_prompt = f"Genera una respuesta final para la pregunta: {user_question}. Usa la información de la herramienta: {tool_output}"
        else:
            final_prompt = user_question

        start = time.time()
        final_answer = self._generate(final_prompt)
        end = time.time()
        trace.add_step(current_step, 'llm', final_prompt, final_answer['text'], end - start, 50, 'ok' if final_answer['error'] is None else 'error')

        return trace

agent = ToolAgent(llm, csv_path)

for pregunta in [
    'Calcula 45 + 81.',
    'Busca información sobre observabilidad.',
    'Consulta el archivo CSV sobre clima.',
    'Explica cómo funciona la batería de un auto eléctrico.',
]:
    print(f'=== Ejecución para pregunta: {pregunta}')
    trace = agent.run(pregunta)
    trace.display()


=== Ejecución para pregunta: Calcula 45 + 81.


,step,tipo,input,output,latencia,tokens,estado
0,1,llm,Decide qué herramienta usar para: Calcula 45 +...,"Para calcular **45 + 81**, puedes usar una de ...",2.35s,30,ok
1,2,tool,45 + 81.,126.0,0.00s,0,ok
2,3,llm,Genera una respuesta final para la pregunta: C...,El resultado de calcular 45 + 81 es **126**.,1.18s,50,ok


=== Ejecución para pregunta: Busca información sobre observabilidad.


,step,tipo,input,output,latencia,tokens,estado
0,1,llm,Decide qué herramienta usar para: Busca inform...,Para buscar información sobre **observabilidad...,2.92s,30,ok
1,2,tool,Busca información sobre observabilidad.,[No se encontró información relevante.],0.00s,0,ok
2,3,llm,Genera una respuesta final para la pregunta: B...,No se encontró información relevante sobre obs...,1.59s,50,ok


=== Ejecución para pregunta: Consulta el archivo CSV sobre clima.


,step,tipo,input,output,latencia,tokens,estado
0,1,llm,Decide qué herramienta usar para: Consulta el ...,"Para consultar un archivo CSV sobre clima, la ...",5.82s,30,ok
1,2,tool,clima.,No hay datos relevantes en el CSV.,0.00s,0,ok
2,3,llm,Genera una respuesta final para la pregunta: C...,Parece que el archivo CSV consultado no contie...,0.88s,50,ok


=== Ejecución para pregunta: Explica cómo funciona la batería de un auto eléctrico.


,step,tipo,input,output,latencia,tokens,estado
0,1,llm,Decide qué herramienta usar para: Explica cómo...,Para explicar cómo funciona la batería de un a...,3.90s,30,ok
1,2,tool,Explica cómo funciona la batería de un auto el...,[No se encontró información relevante.],0.00s,0,ok
2,3,llm,Genera una respuesta final para la pregunta: E...,La batería de un auto eléctrico funciona almac...,1.73s,50,ok


In [ ]:
# Ejemplo de uso del agente de Wikipedia para buscar información durante la trazabilidad
wiki_query = '¿Qué es la observabilidad para agentes LLM?'
wiki_response = agent_executor.invoke({
    'input': wiki_query,
    'chat_history': []
})
print('=== Respuesta del agente de Wikipedia ===')
print(wiki_response['output'])

## Análisis de la traza

Cada fila representa un span o paso del agente. Las columnas permiten detectar: 
- si el agente llamó a una herramienta (`tipo` = tool),
- cuánto tardó cada paso (`latencia`),
- si hubo errores (`estado`),
- qué datos fueron utilizados como entrada y salida.

Esto nos ayuda a identificar ciclos repetitivos, decisiones erróneas o pasos innecesarios en un agente complejo.

: {
: {
: 
3
, 
: 
, 
: 
},
: {
: 
}
: 4,
: 5